# 第一章：图像拼接模型

## 编程实践：手写线性最小二乘 + RANSAC 仿射拼接

| 项目 | 说明 |
|------|------|
| 输入图片 | `stitch1.jpg` / `stitch2.jpg`（来自 Hands-on-CV 参考库第 8 章） |
| 手写核心 | 线性最小二乘（高斯消元解正规方程）、RANSAC、仿射/单应 warp 与融合 |
| 允许调用 | 图像读写、矩阵计算、特征点提取与匹配（OpenCV/numpy） |
| 对比验证 | 与 `np.linalg.lstsq` 求出的仿射解做数值对比 |


## 一、学习目标

1. 掌握**图像拼接**的基本概念和原理。
2. 掌握**线性最小二乘优化算法**及其编程实现。
3. 掌握 **RANSAC** 在外点剔除中的作用，并能手写实现。

### 线性最小二乘

对仿射模型：

```
u = a*x + b*y + tx
v = c*x + d*y + ty
```

把 N 对匹配点写成超定线性方程组 `A p = b`，最小二乘解满足正规方程：

```
(A^T A) p = A^T b
```

本页手写**高斯消元**求解该正规方程，得到仿射参数 `p = [a,b,tx,c,d,ty]^T`。

### RANSAC

1. 随机采样最小点集（仿射 3 对）拟合模型；
2. 统计所有点在该模型下的内点（重投影误差 < 阈值）；
3. 迭代多次，取内点数最多的模型；
4. 用全部内点重新拟合最终模型。

RANSAC 能鲁棒地抵抗错误匹配（外点）的干扰。


## 二、手写约束清单（二阶段）

- ✅ 允许调用：`cv_imread` / `cv_imwrite`（读写）、numpy 矩阵计算（如 SVD）、`cv2.SIFT_create` / `cv2.BFMatcher`（特征提取与匹配）。
- ❌ 其余必须手写：最小二乘求解、RANSAC、仿射/单应变换参数求解、warp 与图像融合。
- 本章统一 `set_random_seed(42)`，保证 RANSAC 随机采样可复现。


In [ ]:
import sys
from pathlib import Path

# 向上查找项目根目录（含 utils.py），并加入 sys.path
ROOT = Path.cwd().resolve()
while not (ROOT / "utils.py").exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("未找到项目根目录 utils.py")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import cv2
import matplotlib.pyplot as plt

from utils import cv_imread, cv_imwrite, set_random_seed, setup_plot_chinese, show_images, compare_results

setup_plot_chinese()
set_random_seed(42)
print(f"OpenCV 版本: {cv2.__version__}")
print(f"当前工作目录: {Path.cwd()}")


In [ ]:
import math


def solve_linear_system(A, b):
    """手写高斯消元（部分主元）求解方阵线性方程组 A x = b。"""
    A = [list(map(float, row)) for row in A]
    b = [float(v) for v in b]
    n = len(b)
    for col in range(n):
        pivot = max(range(col, n), key=lambda r: abs(A[r][col]))
        if abs(A[pivot][col]) < 1e-12:
            return None
        A[col], A[pivot] = A[pivot], A[col]
        b[col], b[pivot] = b[pivot], b[col]
        for r in range(col + 1, n):
            f = A[r][col] / A[col][col]
            for c in range(col, n):
                A[r][c] -= f * A[col][c]
            b[r] -= f * b[col]
    x = [0.0] * n
    for i in range(n - 1, -1, -1):
        s = b[i]
        for j in range(i + 1, n):
            s -= A[i][j] * x[j]
        x[i] = s / A[i][i]
    return x


def estimate_affine_least_squares(src, dst):
    """手写最小二乘估计 2D 仿射变换 dst = M * src，M 为 2x3。"""
    n = len(src)
    # 正规方程 A^T A p = A^T b
    ATA = [[0.0] * 6 for _ in range(6)]
    ATb = [0.0] * 6
    for i in range(n):
        x, y = float(src[i][0]), float(src[i][1])
        u, v = float(dst[i][0]), float(dst[i][1])
        row0 = [x, y, 1.0, 0.0, 0.0, 0.0]
        row1 = [0.0, 0.0, 0.0, x, y, 1.0]
        for r in range(6):
            ATb[r] += row0[r] * u + row1[r] * v
            for c in range(6):
                ATA[r][c] += row0[r] * row0[c] + row1[r] * row1[c]

    p = solve_linear_system(ATA, ATb)
    if p is None:
        return None
    return np.array([[p[0], p[1], p[2]], [p[3], p[4], p[5]]], dtype=np.float64)


def reprojection_error_affine(M, src, dst):
    """计算仿射变换的重投影均方根误差。"""
    err2 = 0.0
    for i in range(len(src)):
        x, y = float(src[i][0]), float(src[i][1])
        u, v = float(dst[i][0]), float(dst[i][1])
        ue = M[0, 0] * x + M[0, 1] * y + M[0, 2]
        ve = M[1, 0] * x + M[1, 1] * y + M[1, 2]
        err2 += (u - ue) ** 2 + (v - ve) ** 2
    return math.sqrt(err2 / len(src))


def ransac_affine(src, dst, threshold=3.0, iterations=500):
    """手写 RANSAC 估计仿射变换，返回 (模型, 内点下标列表)。"""
    n = len(src)
    if n < 3:
        return None, []
    best_M, best_inliers, best_count = None, [], -1
    for _ in range(iterations):
        idx = np.random.choice(n, 3, replace=False)
        M = estimate_affine_least_squares(src[idx], dst[idx])
        if M is None:
            continue
        inliers = []
        for k in range(n):
            x, y = float(src[k][0]), float(src[k][1])
            u, v = float(dst[k][0]), float(dst[k][1])
            ue = M[0, 0] * x + M[0, 1] * y + M[0, 2]
            ve = M[1, 0] * x + M[1, 1] * y + M[1, 2]
            if (u - ue) ** 2 + (v - ve) ** 2 <= threshold * threshold:
                inliers.append(k)
        if len(inliers) > best_count:
            best_count = len(inliers)
            best_inliers = inliers
            best_M = M
    if best_inliers:
        best_M = estimate_affine_least_squares(src[best_inliers], dst[best_inliers])
    return best_M, best_inliers


def invert_matrix3(M):
    """手写 3x3 矩阵求逆（伴随矩阵法）。"""
    M = np.array(M, dtype=np.float64)
    cof = np.zeros((3, 3))
    for i in range(3):
        for j in range(3):
            sub = np.delete(np.delete(M, i, axis=0), j, axis=1)
            cof[i, j] = ((-1) ** (i + j)) * (sub[0, 0] * sub[1, 1] - sub[0, 1] * sub[1, 0])
    det = M[0, 0] * cof[0, 0] + M[0, 1] * cof[0, 1] + M[0, 2] * cof[0, 2]
    return cof.T / det


def warp_affine_manual(image, M, out_w, out_h, border_value=0):
    """手写仿射 warp：反向映射 + 双线性插值。"""
    H = np.vstack([M, [0, 0, 1]])
    H_inv = invert_matrix3(H)
    out = np.full((out_h, out_w, image.shape[2]), border_value, dtype=np.uint8)
    for y in range(out_h):
        for x in range(out_w):
            sx = H_inv[0, 0] * x + H_inv[0, 1] * y + H_inv[0, 2]
            sy = H_inv[1, 0] * x + H_inv[1, 1] * y + H_inv[1, 2]
            sw = H_inv[2, 0] * x + H_inv[2, 1] * y + H_inv[2, 2]
            if abs(sw) < 1e-9:
                continue
            ix, iy = sx / sw, sy / sw
            if ix < 0 or iy < 0 or ix > image.shape[1] - 1 or iy > image.shape[0] - 1:
                continue
            x0, y0 = int(ix), int(iy)
            x1, y1 = min(x0 + 1, image.shape[1] - 1), min(y0 + 1, image.shape[0] - 1)
            fx, fy = ix - x0, iy - y0
            for ch in range(image.shape[2]):
                top = image[y0, x0, ch] * (1 - fx) + image[y0, x1, ch] * fx
                bottom = image[y1, x0, ch] * (1 - fx) + image[y1, x1, ch] * fx
                out[y, x, ch] = int(round(top * (1 - fy) + bottom * fy))
    return out


def estimate_homography_dlt(src, dst):
    """手写归一化 DLT 估计单应矩阵（3x3）。"""
    def normalize(pts):
        c = pts.mean(axis=0)
        d = np.mean(np.linalg.norm(pts - c, axis=1))
        s = math.sqrt(2.0) / d
        T = np.array([[s, 0, -s * c[0]], [0, s, -s * c[1]], [0, 0, 1]])
        return T

    T_src = normalize(src)
    T_dst = normalize(dst)
    A = []
    for (x, y), (u, v) in zip(src, dst):
        xh = T_src @ np.array([x, y, 1.0])
        uh = T_dst @ np.array([u, v, 1.0])
        x, y = xh[0], xh[1]
        u, v = uh[0], uh[1]
        A.append([-x, -y, -1, 0, 0, 0, u * x, u * y, u])
        A.append([0, 0, 0, -x, -y, -1, v * x, v * y, v])
    A = np.array(A)
    _, _, Vt = np.linalg.svd(A)
    H = Vt[-1].reshape(3, 3)
    H = np.linalg.inv(T_dst) @ H @ T_src
    return H / H[2, 2]


def warp_homography_manual(image, H, out_w, out_h, border_value=0):
    """手写单应 warp：反向映射 + 双线性插值。"""
    H_inv = invert_matrix3(H)
    out = np.full((out_h, out_w, image.shape[2]), border_value, dtype=np.uint8)
    for y in range(out_h):
        for x in range(out_w):
            sx = H_inv[0, 0] * x + H_inv[0, 1] * y + H_inv[0, 2]
            sy = H_inv[1, 0] * x + H_inv[1, 1] * y + H_inv[1, 2]
            sw = H_inv[2, 0] * x + H_inv[2, 1] * y + H_inv[2, 2]
            if abs(sw) < 1e-9:
                continue
            ix, iy = sx / sw, sy / sw
            if ix < 0 or iy < 0 or ix > image.shape[1] - 1 or iy > image.shape[0] - 1:
                continue
            x0, y0 = int(ix), int(iy)
            x1, y1 = min(x0 + 1, image.shape[1] - 1), min(y0 + 1, image.shape[0] - 1)
            fx, fy = ix - x0, iy - y0
            for ch in range(image.shape[2]):
                top = image[y0, x0, ch] * (1 - fx) + image[y0, x1, ch] * fx
                bottom = image[y1, x0, ch] * (1 - fx) + image[y1, x1, ch] * fx
                out[y, x, ch] = int(round(top * (1 - fy) + bottom * fy))
    return out


def stitch_two_images(left, right, M):
    """把 right 用 M（2x3 仿射或 3x3 单应）变换到 left 坐标系，并做重叠区平均融合。"""
    h_l, w_l = left.shape[:2]
    h_r, w_r = right.shape[:2]
    if M.shape == (2, 3):
        M3 = np.vstack([M, [0, 0, 1]])
    else:
        M3 = M
    corners = np.array([[0, 0], [w_r, 0], [w_r, h_r], [0, h_r]], dtype=np.float64)
    xs, ys = [], []
    for x, y in corners:
        denom = M3[2, 0] * x + M3[2, 1] * y + M3[2, 2]
        xs.append((M3[0, 0] * x + M3[0, 1] * y + M3[0, 2]) / denom)
        ys.append((M3[1, 0] * x + M3[1, 1] * y + M3[1, 2]) / denom)
    xmin = min(0, w_l, *xs); xmax = max(0, w_l, *xs)
    ymin = min(0, h_l, *ys); ymax = max(0, h_l, *ys)
    pan_w = int(math.ceil(xmax - xmin)); pan_h = int(math.ceil(ymax - ymin))
    T3 = np.array([[1, 0, -xmin], [0, 1, -ymin], [0, 0, 1]])
    M_final = T3 @ M3
    warped = warp_homography_manual(right, M_final, pan_w, pan_h, border_value=0)
    base = np.zeros((pan_h, pan_w, 3), dtype=np.uint8)
    ly0, lx0 = int(-ymin), int(-xmin)
    base[ly0:ly0 + h_l, lx0:lx0 + w_l] = left
    mask_base = base.sum(axis=2) > 0
    mask_warp = warped.sum(axis=2) > 0
    overlap = mask_base & mask_warp
    result = base.copy()
    result[~mask_base] = warped[~mask_base]
    result[overlap] = (base[overlap].astype(np.float32) + warped[overlap].astype(np.float32)) / 2
    return result.astype(np.uint8)


In [ ]:
# 读取两张有重叠的图像
left = cv_imread("stitch1.jpg", cv2.IMREAD_COLOR)
right = cv_imread("stitch2.jpg", cv2.IMREAD_COLOR)
assert left is not None and right is not None, "读取 stitch1.jpg / stitch2.jpg 失败"
show_images([left, right], ["左图", "右图"], figsize=(9, 4))


In [ ]:
# 特征点提取与匹配（允许调用 OpenCV）
gray_l = cv2.cvtColor(left, cv2.COLOR_BGR2GRAY)
gray_r = cv2.cvtColor(right, cv2.COLOR_BGR2GRAY)
sift = cv2.SIFT_create()
kp_l, des_l = sift.detectAndCompute(gray_l, None)
kp_r, des_r = sift.detectAndCompute(gray_r, None)

bf = cv2.BFMatcher(cv2.NORM_L2)
knns = bf.knnMatch(des_l, des_r, k=2)
good = [m for m, n in knns if m.distance < 0.75 * n.distance]
print(f"良好匹配点对数: {len(good)}")

src = np.float32([kp_r[m.trainIdx].pt for m in good])   # 右图坐标
dst = np.float32([kp_l[m.queryIdx].pt for m in good])   # 左图坐标
print("src(右图) -> dst(左图)")


In [ ]:
# 手写 RANSAC + 最小二乘求仿射变换
M_affine, inliers = ransac_affine(src, dst, threshold=3.0, iterations=500)
print(f"RANSAC 内点数: {len(inliers)} / {len(good)}")

if M_affine is not None and len(inliers) >= 3:
    src_in = src[inliers]
    dst_in = dst[inliers]
    print("仿射变换矩阵 M (右图 -> 左图):")
    print(M_affine)
    print(f"内点重投影 RMSE: {reprojection_error_affine(M_affine, src_in, dst_in):.4f} px")

    # 与 numpy lstsq 对比验证（矩阵计算允许；仅验证）
    A = []
    b = []
    for (x, y), (u, v) in zip(src_in, dst_in):
        A.append([x, y, 1, 0, 0, 0]); b.append(u)
        A.append([0, 0, 0, x, y, 1]); b.append(v)
    p_np, *_ = np.linalg.lstsq(np.array(A), np.array(b), rcond=None)
    M_np = np.array([[p_np[0], p_np[1], p_np[2]], [p_np[3], p_np[4], p_np[5]]])
    print("与 np.linalg.lstsq 的最大参数差:",
          float(np.max(np.abs(M_affine - M_np))))

    panorama = stitch_two_images(left, right, M_affine)
    cv_imwrite("stitched_panorama.jpg", panorama)
    show_images([left, right, panorama], ["左图", "右图", "仿射拼接结果"], figsize=(13, 4))
else:
    print("仿射拼接失败：内点不足")


In [ ]:
# 拓展（可选）：改用单应变换拼接
if len(good) >= 4:
    # 手写 RANSAC + DLT 单应
    best_H, best_in = None, []
    for _ in range(500):
        idx = np.random.choice(len(good), 4, replace=False)
        H = estimate_homography_dlt(src[idx], dst[idx])
        inl = []
        for k in range(len(good)):
            x, y = src[k]
            u, v = dst[k]
            denom = H[2, 0] * x + H[2, 1] * y + H[2, 2]
            if abs(denom) < 1e-9:
                continue
            ue = (H[0, 0] * x + H[0, 1] * y + H[0, 2]) / denom
            ve = (H[1, 0] * x + H[1, 1] * y + H[1, 2]) / denom
            if (u - ue) ** 2 + (v - ve) ** 2 <= 3.0 ** 2:
                inl.append(k)
        if len(inl) > len(best_in):
            best_in = inl; best_H = H
    H = estimate_homography_dlt(src[best_in], dst[best_in])
    print(f"单应内点数: {len(best_in)}")
    pano_h = stitch_two_images(left, right, H)
    cv_imwrite("stitched_homography.jpg", pano_h)
    show_images([panorama, pano_h], ["仿射拼接", "单应拼接"], figsize=(12, 4))


## 三、结果与参数分析

- RANSAC 内点数应明显大于外点数；若内点率过低，可调大 `threshold` 或调低 `iterations` 前先提高匹配质量（调小比值阈值）。
- 仿射模型只表达 6 自由度，适合近似平面、无强透视的拼接；单应模型有 8 自由度，更适合有透视变形的拼接。
- 重叠区用简单平均融合；更工程化的做法是加权融合（距离中心越近权重越大）或拉普拉斯金字塔融合。

**易错点**
1. 注意 `src`/`dst` 方向：本页是"右图坐标 → 左图坐标"，方向搞反会导致拼接错位。
2. RANSAC 最小点集：仿射至少 3 对，单应至少 4 对。
3. warp 输出画布要先计算四个角点变换后的包围盒，否则图像被裁掉。


## 四、科研规范小结

1. **估计器与验证分离**：`estimate_*` 只负责拟合，`reprojection_error_*` 只负责评估。
2. **随机可复现**：RANSAC 前统一 `set_random_seed(42)`。
3. **手写底层 + 库验证**：手写高斯消元求最小二乘，再用 `np.linalg.lstsq` 验证参数一致性。
4. **内点统计透明**：报告内点数、内点率与重投影 RMSE，而不是只贴一张图。


## 五、练习：改变 RANSAC 阈值观察内点率

**要求**：分别用 `threshold=1.0 / 3.0 / 6.0` 运行 `ransac_affine`，打印内点数与内点率，说明阈值对内点筛选和最终拼接的影响。


In [ ]:
# ==================== 练习解决方案 ====================
for thr in [1.0, 3.0, 6.0]:
    M, inl = ransac_affine(src, dst, threshold=thr, iterations=300)
    print(f"threshold={thr:.1f} -> 内点数 {len(inl)}/{len(good)} ({100*len(inl)/len(good):.1f}%)")
